In [8]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset.shape

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'train': (287113, 3), 'validation': (13368, 3), 'test': (11490, 3)}

In [9]:
sample = dataset["train"][0]

print(sample.keys())
print("\nARTICLE:\n", sample["article"][:1000])
print("\nHIGHLIGHTS:\n", sample["highlights"])

dict_keys(['article', 'highlights', 'id'])

ARTICLE:
 LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his numbe

In [10]:
for i in range(3):
    article = dataset["train"][i]["article"]
    summary = dataset["train"][i]["highlights"]
    print(f"Sample {i}")
    print("Article length:", len(article.split()))
    print("Summary length:", len(summary.split()))
    print("-" * 50)

small_train = dataset["train"].select(range(200))
small_val = dataset["validation"].select(range(50))

Sample 0
Article length: 455
Summary length: 41
--------------------------------------------------
Sample 1
Article length: 698
Summary length: 49
--------------------------------------------------
Sample 2
Article length: 743
Summary length: 43
--------------------------------------------------


In [22]:
!pip install transformers sentencepiece

In [13]:
from transformers import T5Tokenizer

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [11]:
sample_article = small_train[0]["article"]
sample_summary = small_train[0]["highlights"]

input_text = "summarize: " + sample_article
target_text = sample_summary

print(input_text[:300])
print()
print(target_text)

summarize: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To t

Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [14]:
inputs = tokenizer(
    input_text,
    max_length=512,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

targets = tokenizer(
    target_text,
    max_length=64,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

print("Input IDs shape:", inputs["input_ids"].shape)
print("Target IDs shape:", targets["input_ids"].shape)

Input IDs shape: torch.Size([1, 512])
Target IDs shape: torch.Size([1, 64])


In [15]:
def preprocess(example):
    input_text = "summarize: " + example["article"]
    target_text = example["highlights"]

    inputs = tokenizer(
        input_text,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    targets = tokenizer(
        target_text,
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = targets["input_ids"]

    # Replace padding token id with -100
    labels = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in labels
    ]

    inputs["labels"] = labels

    return inputs

In [16]:
tokenized_train = small_train.map(preprocess, batched=False)
tokenized_val = small_val.map(preprocess, batched=False)


tokenized_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_val.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [28]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [15]:
print(type(model))

<class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


In [29]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./t5_summarizer",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=1,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [32]:
test_article = small_val[0]["article"]
true_summary = small_val[0]["highlights"]

input_text = "summarize: " + test_article

In [33]:
inputs = tokenizer(
    input_text,
    return_tensors="pt",
    max_length=256,
    truncation=True
)

In [34]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = {k: v.to(device) for k, v in inputs.items()}

In [35]:
output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=32,
    num_beams=4,
    early_stopping=True
)

In [36]:
generated_summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("ARTICLE:\n")
print(test_article[:1000])

print("\nTRUE SUMMARY:\n")
print(true_summary)

print("\nGENERATED SUMMARY:\n")
print(generated_summary)

ARTICLE:

(CNN)Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data. It resulted in six patients receiving transplants. That surprised and wowed her. "I thought I was going to help this one person who I don't know, but the fact that so many people can have a life extension, that's pretty big," Broussard told CNN affiliate KGO. She may feel guided in her generosity by a higher power. "Thanks for all the support and prayers," a comment on a Facebook page in her name read. "I know this entire journey is much bigger than all of us. I also know I'm just the messenger." CNN cannot verify the authenticity of the page. But the power that multiplied Broussard's gift was data processing of genetic profiles from donor-recipient pairs. It works on a simple swapping principle but takes it to a much higher level, according to California Pacific Medic

In [37]:
for i in range(3):
    test_article = small_val[i]["article"]
    true_summary = small_val[i]["highlights"]

    input_text = "summarize: " + test_article

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=32,
        num_beams=4,
        early_stopping=True
    )

    generated_summary = tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"\n{'='*80}")
    print(f"SAMPLE {i}")
    print(f"{'='*80}")
    print("TRUE SUMMARY:\n", true_summary)
    print("\nGENERATED SUMMARY:\n", generated_summary)


SAMPLE 0
TRUE SUMMARY:
 Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

GENERATED SUMMARY:
 Zully Broussard gave one of her kidneys to a stranger. it resulted in six patients receiving transplants. she

SAMPLE 1
TRUE SUMMARY:
 The 20th MLS season begins this weekend .
League has changed dramatically since its inception in 1996 .
Some question whether rules regarding salary caps and transfers need to change .

GENERATED SUMMARY:
 San Jose Clash and DC United strode out in front of 31,683 expectant fans at the Spartan Stadium in San Jose, California

SAMPLE 2
TRUE SUMMARY:
 Bafetimbi Gomis collapses within 10 minutes of kickoff at Tottenham .
But he reportedly left the pitch conscious and wearing an oxygen mask .
Gomis later said that he was "feeling well"
The incident came three years after Fabrice Muamba collapsed at White Hart Lane .

GENERATED SUMMARY:
 Bafetimbi Gomis is now "feeling wel

In [38]:
results = []

for i in range(10):
    test_article = small_val[i]["article"]
    true_summary = small_val[i]["highlights"]

    input_text = "summarize: " + test_article

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=32,
        num_beams=4,
        early_stopping=True
    )

    generated_summary = tokenizer.decode(output[0], skip_special_tokens=True)

    results.append({
        "article": test_article,
        "true_summary": true_summary,
        "generated_summary": generated_summary
    })

In [40]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df.head(10)

,article,true_summary,generated_summary
0,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,Zully Broussard gave one of her kidneys to a s...
1,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,San Jose Clash and DC United strode out in fro...
2,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,"Bafetimbi Gomis is now ""feeling well"" after co..."
3,(CNN)It was an act of frustration perhaps more...,Rory McIlroy throws club into water at WGC Cad...,Rory McIlroy pulls his second shot on the eigh...
4,(CNN)A Pennsylvania community is pulling toget...,"Cayman Naib, 13, hasn't been heard from since ...","cayman Naib, 13, was last seen wearing a gray ..."
5,(CNN)My vote for Father of the Year goes to Cu...,Ruben Navarrette: Schilling deserves praise fo...,Curt Schilling recently fired off a series of ...
6,"(CNN)Another one for the ""tourists behaving ba...",Two American women arrested for carving initia...,two american women have reportedly been arrest...
7,(CNN)Following last year's successful U.K. tou...,It will be a first time for the tour stateside...,Prince and 3rdEyeGirl are bringing the Hit & R...
8,(CNN)A shooting at a bar popular with expatria...,A jihadist group claims responsibility in an a...,"authorities call the shooting a ""criminal and ..."
9,(CNN)Manchester United defender Jonny Evans an...,Alleged incident happened in match at St James...,manchester united defender Jonny Evans and pap...


In [41]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6cf11aab04f80de30c240195d43dbc421f071343bec5104f96c38f6fc50e7674
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [42]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for i in range(len(results_df)):
    scores = scorer.score(
        results_df.loc[i, "true_summary"],
        results_df.loc[i, "generated_summary"]
    )
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print("Average ROUGE-1:", sum(rouge1_scores) / len(rouge1_scores))
print("Average ROUGE-2:", sum(rouge2_scores) / len(rouge2_scores))
print("Average ROUGE-L:", sum(rougeL_scores) / len(rougeL_scores))

Average ROUGE-1: 0.290397731510767
Average ROUGE-2: 0.12307727889462398
Average ROUGE-L: 0.22878982352740004


In [43]:
output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=50,
    min_length=15,
    num_beams=4,
    early_stopping=True
)

In [44]:
!pip install --upgrade google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.47.0
    Uninstalling google-auth-2.47.0:
      Successfully uninstalled google-auth-2.47.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.68.0
    Uninstalling google-genai-1.68.0:
      Successfully uninstalled google-genai-1.68.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
PROJECT_ID = "dsml-492506"
LOCATION = "us-central1"

In [3]:
from google import genai
from google.genai.types import HttpOptions

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1")
)

In [4]:
MODEL_NAME = "gemini-2.5-flash"

In [5]:
def build_prompt(article_text):
    return f"""
You are a news summarization assistant.

Summarize the following news article in 2 to 3 concise sentences.
Focus only on the key facts.
Do not add information that is not in the article.

Article:
{article_text}
"""

In [17]:
article_text = small_val[0]["article"]
true_summary = small_val[0]["highlights"]
prompt = build_prompt(article_text)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt,
)

vertex_summary = response.text

print("TRUE SUMMARY:\n", true_summary)
print("\nVERTEX SUMMARY:\n", vertex_summary)

TRUE SUMMARY:
 Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

VERTEX SUMMARY:
 Zully Broussard's altruistic kidney donation to a stranger initiated a rare "super swap" chain, leading to six patients receiving transplants. This complex chain, involving 12 surgeries, was rapidly organized using a specialized program called MatchGrid, which efficiently processes genetic profiles for donor-recipient matching. The system significantly increased matching options and allowed the process to be completed in about three weeks.


In [18]:
vertex_results = []

for i in range(5):
    article_text = small_val[i]["article"]
    true_summary = small_val[i]["highlights"]
    prompt = build_prompt(article_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
    )

    vertex_summary = response.text

    vertex_results.append({
        "article": article_text,
        "true_summary": true_summary,
        "vertex_summary": vertex_summary,
    })

In [19]:
import pandas as pd

vertex_results_df = pd.DataFrame(vertex_results)
vertex_results_df.head()

,article,true_summary,vertex_summary
0,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,Zully Broussard's altruistic kidney donation t...
1,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"Major League Soccer (MLS) launched on April 6,..."
2,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,French striker Bafetimbi Gomis collapsed durin...
3,(CNN)It was an act of frustration perhaps more...,Rory McIlroy throws club into water at WGC Cad...,"During the WGC Cadillac Championship, Rory McI..."
4,(CNN)A Pennsylvania community is pulling toget...,"Cayman Naib, 13, hasn't been heard from since ...",Thirteen-year-old Cayman Naib has been missing...


In [20]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for i in range(len(vertex_results_df)):
    scores = scorer.score(
        vertex_results_df.loc[i, "true_summary"],
        vertex_results_df.loc[i, "vertex_summary"]
    )
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print("Vertex Average ROUGE-1:", sum(rouge1_scores) / len(rouge1_scores))
print("Vertex Average ROUGE-2:", sum(rouge2_scores) / len(rouge2_scores))
print("Vertex Average ROUGE-L:", sum(rougeL_scores) / len(rougeL_scores))

Vertex Average ROUGE-1: 0.2651773431744131
Vertex Average ROUGE-2: 0.06348179271708684
Vertex Average ROUGE-L: 0.17920426703302406
